In [1]:
#!pip3 install twilio

In [2]:
#!pip3 install pandas

In [3]:
#!pip3 install tqdm 

In [4]:
import sys
print(sys.executable)

!{sys.executable} -m pip show twilio

/home/ingdavidhoyosgil/anaconda3/envs/de_weather/bin/python
Name: twilio
Version: 9.10.9
Summary: Twilio API client and TwiML generator
Home-page: https://github.com/twilio/twilio-python/
Author: Twilio
Author-email: 
License: MIT
Location: /home/ingdavidhoyosgil/anaconda3/envs/de_weather/lib/python3.14/site-packages
Requires: aiohttp, aiohttp-retry, PyJWT, requests
Required-by: 


In [5]:
import twilio_config
print(dir(twilio_config))

['API_KEY_WAPI', 'PHONE_NUMBER', 'PRIVATE_NUMBER', 'TWILIO_ACCOUNT_SID', 'TWILIO_AUTH_TOKEN', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__']


In [6]:
import os
from twilio.rest import Client
from twilio_config import TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN, PRIVATE_NUMBER, API_KEY_WAPI
import time

from requests import Request, Session
from requests.exceptions import ConnectionError, Timeout, TooManyRedirects
import json

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

from datetime import datetime 


## URL

In [7]:
query = 'Buenaventura'
api_key = API_KEY_WAPI

url_weather = 'https://api.weatherapi.com/v1/forecast.json?key='+api_key+'&q='+query+'&days=1&aqi=no&alerts=no'

In [8]:
response = requests.get(url_weather).json()

In [9]:
response

{'location': {'name': 'Buenaventura',
  'region': 'Valle del Cauca',
  'country': 'Colombia',
  'lat': 3.8933,
  'lon': -77.0697,
  'tz_id': 'America/Bogota',
  'localtime_epoch': 1784500922,
  'localtime': '2026-07-19 17:42'},
 'current': {'last_updated_epoch': 1784500200,
  'last_updated': '2026-07-19 17:30',
  'temp_c': 26.2,
  'temp_f': 79.2,
  'is_day': 1,
  'condition': {'text': 'Partly cloudy',
   'icon': '//cdn.weatherapi.com/weather/64x64/day/116.png',
   'code': 1003},
  'wind_mph': 2.9,
  'wind_kph': 4.7,
  'wind_degree': 203,
  'wind_dir': 'SSW',
  'pressure_mb': 1009.0,
  'pressure_in': 29.8,
  'precip_mm': 1.1,
  'precip_in': 0.04,
  'humidity': 94,
  'cloud': 50,
  'feelslike_c': 30.1,
  'feelslike_f': 86.2,
  'windchill_c': 25.5,
  'windchill_f': 78.0,
  'heatindex_c': 28.8,
  'heatindex_f': 83.9,
  'dewpoint_c': 24.0,
  'dewpoint_f': 75.3,
  'vis_km': 10.0,
  'vis_miles': 6.0,
  'uv': 0.3,
  'gust_mph': 4.1,
  'gust_kph': 6.7,
  'will_it_rain': 1,
  'chance_of_rain': 7

In [10]:
response.keys()

dict_keys(['location', 'current', 'forecast'])

In [11]:
response['forecast']

{'forecastday': [{'date': '2026-07-19',
   'date_epoch': 1784419200,
   'day': {'maxtemp_c': 27.3,
    'maxtemp_f': 81.1,
    'mintemp_c': 23.7,
    'mintemp_f': 74.6,
    'avgtemp_c': 24.9,
    'avgtemp_f': 76.8,
    'maxwind_mph': 7.2,
    'maxwind_kph': 11.5,
    'totalprecip_mm': 24.59,
    'totalprecip_in': 0.97,
    'totalsnow_cm': 0.0,
    'avgvis_km': 9.9,
    'avgvis_miles': 6.0,
    'avghumidity': 92,
    'daily_will_it_rain': 1,
    'daily_chance_of_rain': 90,
    'daily_will_it_snow': 0,
    'daily_chance_of_snow': 0,
    'condition': {'text': 'Light rain shower',
     'icon': '//cdn.weatherapi.com/weather/64x64/day/353.png',
     'code': 1240},
    'uv': 7.1},
   'astro': {'sunrise': '06:05 AM',
    'sunset': '06:24 PM',
    'moonrise': '10:35 AM',
    'moonset': '10:54 PM',
    'moon_phase': 'First Quarter',
    'moon_illumination': 33,
    'is_moon_up': 1,
    'is_sun_up': 1},
   'hour': [{'time_epoch': 1784437200,
     'time': '2026-07-19 00:00',
     'temp_c': 23.9,
  

In [12]:
response['forecast'].keys()

dict_keys(['forecastday'])

In [13]:
response['forecast']['forecastday']

[{'date': '2026-07-19',
  'date_epoch': 1784419200,
  'day': {'maxtemp_c': 27.3,
   'maxtemp_f': 81.1,
   'mintemp_c': 23.7,
   'mintemp_f': 74.6,
   'avgtemp_c': 24.9,
   'avgtemp_f': 76.8,
   'maxwind_mph': 7.2,
   'maxwind_kph': 11.5,
   'totalprecip_mm': 24.59,
   'totalprecip_in': 0.97,
   'totalsnow_cm': 0.0,
   'avgvis_km': 9.9,
   'avgvis_miles': 6.0,
   'avghumidity': 92,
   'daily_will_it_rain': 1,
   'daily_chance_of_rain': 90,
   'daily_will_it_snow': 0,
   'daily_chance_of_snow': 0,
   'condition': {'text': 'Light rain shower',
    'icon': '//cdn.weatherapi.com/weather/64x64/day/353.png',
    'code': 1240},
   'uv': 7.1},
  'astro': {'sunrise': '06:05 AM',
   'sunset': '06:24 PM',
   'moonrise': '10:35 AM',
   'moonset': '10:54 PM',
   'moon_phase': 'First Quarter',
   'moon_illumination': 33,
   'is_moon_up': 1,
   'is_sun_up': 1},
  'hour': [{'time_epoch': 1784437200,
    'time': '2026-07-19 00:00',
    'temp_c': 23.9,
    'temp_f': 75.0,
    'is_day': 0,
    'condition'

In [14]:
response['forecast']['forecastday'][0].keys()

dict_keys(['date', 'date_epoch', 'day', 'astro', 'hour'])

In [15]:
len(response['forecast']['forecastday'][0]['hour'])

24

In [16]:
response['forecast']['forecastday'][0]['hour'][15]

{'time_epoch': 1784491200,
 'time': '2026-07-19 15:00',
 'temp_c': 26.1,
 'temp_f': 78.9,
 'is_day': 1,
 'condition': {'text': 'Light rain shower',
  'icon': '//cdn.weatherapi.com/weather/64x64/day/353.png',
  'code': 1240},
 'wind_mph': 4.3,
 'wind_kph': 6.8,
 'wind_degree': 229,
 'wind_dir': 'SW',
 'pressure_mb': 1009.0,
 'pressure_in': 29.79,
 'precip_mm': 1.64,
 'precip_in': 0.06,
 'snow_cm': 0.0,
 'humidity': 90,
 'cloud': 86,
 'feelslike_c': 29.6,
 'feelslike_f': 85.3,
 'windchill_c': 26.1,
 'windchill_f': 78.9,
 'heatindex_c': 29.6,
 'heatindex_f': 85.3,
 'dewpoint_c': 24.2,
 'dewpoint_f': 75.6,
 'will_it_rain': 1,
 'chance_of_rain': 74,
 'will_it_snow': 0,
 'chance_of_snow': 0,
 'vis_km': 10.0,
 'vis_miles': 6.0,
 'gust_mph': 5.8,
 'gust_kph': 9.4,
 'uv': 2.5,
 'short_rad': 337.04,
 'diff_rad': 197.13,
 'dni': 11.59,
 'gti': 336.94}

In [17]:
response['forecast']['forecastday'][0]['hour'][15]['time']

'2026-07-19 15:00'

In [18]:
response['forecast']['forecastday'][0]['hour'][15]['time'].split()[0] # Date

'2026-07-19'

In [19]:
int(response['forecast']['forecastday'][0]['hour'][15]['time'].split()[1].split(':')[0]) # Hour

15

In [20]:
response['forecast']['forecastday'][0]['hour'][15]['condition']['text'] # Condition

'Light rain shower'

In [21]:
response['forecast']['forecastday'][0]['hour'][15]['temp_c'] # Temperature

26.1

In [22]:
response['forecast']['forecastday'][0]['hour'][15]['will_it_rain'] 

1

In [23]:
response['forecast']['forecastday'][0]['hour'][15]['chance_of_rain'] 

74

## Dataframe

In [24]:
def get_forecast(response, i):
    forecast = response['forecast']['forecastday'][0]['hour'][i]
    
    date = forecast['time'].split()[0]
    hour = int(forecast['time'].split()[1].split(':')[0])
    condition = forecast['condition']['text']
    temperature_c = forecast['temp_c']
    will_it_rain = forecast['will_it_rain'] 
    chance_of_rain = forecast['chance_of_rain'] 

    return date, hour, condition, temperature_c, will_it_rain, chance_of_rain
    

In [25]:
data = []
size = len(response['forecast']['forecastday'][0]['hour'])

for i in tqdm(range(size), colour = 'green'):
    data.append(get_forecast(response, i))

100%|████████████████████████████| 24/24 [00:00<00:00, 145888.83it/s]


In [26]:
data

[('2026-07-19', 0, 'Light rain shower', 23.9, 0, 62),
 ('2026-07-19', 1, 'Cloudy', 23.8, 0, 26),
 ('2026-07-19', 2, 'Cloudy', 23.7, 0, 25),
 ('2026-07-19', 3, 'Patchy rain nearby', 23.8, 0, 26),
 ('2026-07-19', 4, 'Light rain shower', 23.8, 0, 60),
 ('2026-07-19', 5, 'Light rain shower', 23.8, 0, 66),
 ('2026-07-19', 6, 'Light rain shower', 23.7, 1, 79),
 ('2026-07-19', 7, 'Light rain shower', 24.1, 1, 75),
 ('2026-07-19', 8, 'Light rain shower', 25.1, 0, 41),
 ('2026-07-19', 9, 'Light rain shower', 26.0, 0, 55),
 ('2026-07-19', 10, 'Light rain shower', 27.0, 0, 65),
 ('2026-07-19', 11, 'Light rain shower', 27.3, 0, 59),
 ('2026-07-19', 12, 'Light rain shower', 26.9, 1, 72),
 ('2026-07-19', 13, 'Light rain shower', 26.5, 1, 71),
 ('2026-07-19', 14, 'Light rain shower', 26.3, 1, 74),
 ('2026-07-19', 15, 'Light rain shower', 26.1, 1, 74),
 ('2026-07-19', 16, 'Light rain shower', 25.8, 1, 75),
 ('2026-07-19', 17, 'Partly cloudy', 26.2, 1, 74),
 ('2026-07-19', 18, 'Light rain shower', 24.5

In [27]:
col = ['Date', 'Hour', 'Condition', 'Temperature(C°)', 'Will_it_rain', 'Chance_of_rain']
df = pd.DataFrame(data, columns = col)
df

,Date,Hour,Condition,Temperature(C°),Will_it_rain,Chance_of_rain
0,2026-07-19,0,Light rain shower,23.9,0,62
1,2026-07-19,1,Cloudy,23.8,0,26
2,2026-07-19,2,Cloudy,23.7,0,25
3,2026-07-19,3,Patchy rain nearby,23.8,0,26
4,2026-07-19,4,Light rain shower,23.8,0,60
5,2026-07-19,5,Light rain shower,23.8,0,66
6,2026-07-19,6,Light rain shower,23.7,1,79
7,2026-07-19,7,Light rain shower,24.1,1,75
8,2026-07-19,8,Light rain shower,25.1,0,41
9,2026-07-19,9,Light rain shower,26.0,0,55


In [28]:
START_HOUR = 5
END_HOUR = 22

df_rain = df[
    (df['Will_it_rain'] == 1) & 
    (df['Hour'].between(START_HOUR, END_HOUR))
    ]
df_rain = df_rain[['Hour', 'Condition', 'Chance_of_rain']]
#df_rain.set_index('Hour', inplace = True)
df_rain

,Hour,Condition,Chance_of_rain
6,6,Light rain shower,79
7,7,Light rain shower,75
12,12,Light rain shower,72
13,13,Light rain shower,71
14,14,Light rain shower,74
15,15,Light rain shower,74
16,16,Light rain shower,75
17,17,Partly cloudy,74
18,18,Light rain shower,74
21,21,Light rain,76


## Message from Twilio

In [29]:
#message_string = '\nHello! \n\n\n Forecast for today '+ df['Date'][0] + ' in '+ query + ' is: \n\n\n' + str(df_rain)

message_string = (
    f'🌧️ Rain expected today in {query}\n'
    f'📅 {df['Date'][0]}\n\n'
)

for _, row in df_rain.iterrows():
    message_string += (
        f'🕒 {row['Hour']:02d}:00 - '
        f'{row['Condition']} '
        f'({row['Chance_of_rain']}%)\n'
    )
    
message_string

'🌧️ Rain expected today in Buenaventura\n📅 2026-07-19\n\n🕒 06:00 - Light rain shower (79%)\n🕒 07:00 - Light rain shower (75%)\n🕒 12:00 - Light rain shower (72%)\n🕒 13:00 - Light rain shower (71%)\n🕒 14:00 - Light rain shower (74%)\n🕒 15:00 - Light rain shower (74%)\n🕒 16:00 - Light rain shower (75%)\n🕒 17:00 - Partly cloudy (74%)\n🕒 18:00 - Light rain shower (74%)\n🕒 21:00 - Light rain (76%)\n🕒 22:00 - Light rain shower (79%)\n'

In [30]:
time.sleep(2)
account_sid = TWILIO_ACCOUNT_SID
auth_token = TWILIO_AUTH_TOKEN

client = Client(account_sid, auth_token)

if df["Will_it_rain"].sum() == 0:
    print('No rain expected today. No message will be sent')

else:
    message = client.messages \
                    .create(
                        body = message_string,
                       from_ = "whatsapp:+14155238886",
                        to = 'whatsapp:'+ PRIVATE_NUMBER
                    )

    print('Message acknowledged ' + message.sid)
    print(message.status)

Message acknowledged SM0e5e2711b84c28b8e4dd7ff9d5855192
queued
